### Basic Installations for LoRA (Instruction Fine-Tuning)
We will install the core libraries needed for parameter-efficient fine-tuning (PEFT), model loading, and training.

In [ ]:
!pip install -q -U nemo-automodel transformers datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.4/772.4 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.

In [1]:
import json
from sklearn.model_selection import train_test_split

INPUT_FILE = "/content/drive/MyDrive/IFT/sft_pairs.jsonl"
SEED = 42

with open(INPUT_FILE, encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

sources = sorted({row["source"] for row in data})

train_s, temp_s = train_test_split(
    sources, test_size=0.2, random_state=SEED
)
val_s, test_s = train_test_split(
    temp_s, test_size=0.5, random_state=SEED
)

splits = {
    "train": set(train_s),
    "validation": set(val_s),
    "test": set(test_s),
}

for name, source_set in splits.items():
    rows = [r for r in data if r["source"] in source_set]
    with open(f"/content/drive/MyDrive/IFT/{name}.jsonl", "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"{name}: {len(rows)} examples, {len(source_set)} papers")

assert not (splits["train"] & splits["validation"])
assert not (splits["train"] & splits["test"])
assert not (splits["validation"] & splits["test"])

print("Verified: no paper leakage.")

train: 2808 examples, 152 papers
validation: 273 examples, 19 papers
test: 320 examples, 20 papers
Verified: no paper leakage.


In [ ]:
import json

for split in ["train", "validation", "test"]:
    with open(f"/content/drive/MyDrive/IFT/{split}.jsonl", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    required = ["source", "instruction", "input", "output"]
    invalid = [
        i for i, r in enumerate(rows)
        if any(not isinstance(r.get(k), str) or not r[k].strip()
               for k in required)
    ]

    print(f"{split}: {len(rows)} rows | Invalid: {len(invalid)}")

train: 2808 rows | Invalid: 79
validation: 273 rows | Invalid: 4
test: 320 rows | Invalid: 2


In [ ]:
import json

for split in ["train", "validation", "test"]:
    with open(f"/content/drive/MyDrive/IFT/{split}.jsonl", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    print(f"\n--- {split} ---")
    count = 0

    for i, r in enumerate(rows):
        missing = [
            k for k in ["source", "instruction", "input", "output"]
            if not isinstance(r.get(k), str) or not r[k].strip()
        ]
        if missing and count < 5:
            print(f"Row {i}: missing/empty {missing}")
            count += 1

    print("Total invalid:", sum(
        any(not isinstance(r.get(k), str) or not r[k].strip()
            for k in ["source", "instruction", "input", "output"])
        for r in rows
    ))


--- train ---
Row 4: missing/empty ['input']
Row 12: missing/empty ['input']
Row 13: missing/empty ['input']
Row 14: missing/empty ['input']
Row 15: missing/empty ['input']
Total invalid: 79

--- validation ---
Row 57: missing/empty ['input']
Row 101: missing/empty ['input']
Row 202: missing/empty ['input']
Row 204: missing/empty ['input']
Total invalid: 4

--- test ---
Row 259: missing/empty ['input']
Row 287: missing/empty ['input']
Total invalid: 2


In [ ]:
import json

for split in ["train", "validation", "test"]:
    path = f"/content/drive/MyDrive/IFT/{split}.jsonl"

    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    valid = [
        r for r in rows
        if all(
            isinstance(r.get(k), str) and r[k].strip()
            for k in ["source", "instruction", "input", "output"]
        )
    ]

    with open(path, "w", encoding="utf-8") as f:
        for r in valid:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"{split}: kept {len(valid)}, removed {len(rows) - len(valid)}")

train: kept 2729, removed 79
validation: kept 269, removed 4
test: kept 318, removed 2


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 1
    ), "GB")
else:
    print("No CUDA GPU detected")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


### Step 1: Load Base Model and Tokenizer with Quantization
We'll load the base model in 4-bit precision to save VRAM using `BitsAndBytesConfig`.

In [ ]:
%%writefile /content/drive/MyDrive/IFT/finetune_config.yaml
recipe: TrainFinetuneRecipeForNextTokenPrediction

model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: Qwen/Qwen3-1.7B-Base
  torch_dtype: bfloat16

step_scheduler:
  global_batch_size: 4
  local_batch_size: 1
  num_epochs: 2
  ckpt_every_steps: 100
  val_every_steps: 100
  max_steps: 700

distributed:
  strategy: fsdp2
  dp_size: 1
  tp_size: 1
  cp_size: 1

dist_env:
  backend: nccl
  timeout_minutes: 10

peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  target_modules: "*_proj"
  dim: 16
  alpha: 32
  dropout: 0.05

dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: /content/drive/MyDrive/IFT/train.jsonl
  column_mapping:
    context: input
    question: instruction
    answer: output
  answer_only_loss_mask: true
  seq_length: 512
  truncation: true

validation_dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: /content/drive/MyDrive/IFT/validation.jsonl
  column_mapping:
    context: input
    question: instruction
    answer: output
  answer_only_loss_mask: true
  seq_length: 512
  truncation: true

dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn:
    _target_: nemo_automodel.components.datasets.utils.default_collater
  shuffle: true

validation_dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn:
    _target_: nemo_automodel.components.datasets.utils.default_collater
  shuffle: false

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

optimizer:
  _target_: torch.optim.AdamW
  lr: 0.00002
  weight_decay: 0.01

checkpoint:
  enabled: true
  checkpoint_dir: /content/drive/MyDrive/IFT/qwen3_1.7b_lora

Writing /content/drive/MyDrive/IFT/finetune_config.yaml


In [ ]:
from pathlib import Path
import yaml

config_path = Path("/content/drive/MyDrive/IFT/finetune_config.yaml")
config = yaml.safe_load(config_path.read_text())

print("YAML valid:", isinstance(config, dict))
print("Train file exists:", Path("/content/drive/MyDrive/IFT/train.jsonl").exists())
print("Validation file exists:", Path("/content/drive/MyDrive/IFT/validation.jsonl").exists())
print("Config sections:", list(config.keys()))

YAML valid: True
Train file exists: True
Validation file exists: True
Config sections: ['recipe', 'model', 'step_scheduler', 'distributed', 'dist_env', 'peft', 'dataset', 'validation_dataset', 'dataloader', 'validation_dataloader', 'loss_fn', 'optimizer', 'checkpoint']


In [ ]:
!automodel finetune_config.yaml

File '/content/finetune_config.yaml' was not found.
Traceback (most recent call last):
  File "/usr/local/bin/automodel", line 8, in <module>
    sys.exit(main())
             ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/nemo_automodel/cli/app.py", line 107, in main
    config = load_yaml(config_path)
  File "/usr/local/lib/python3.13/dist-packages/nemo_automodel/cli/utils.py", line 74, in load_yaml
    raise e
  File "/usr/local/lib/python3.13/dist-packages/nemo_automodel/cli/utils.py", line 70, in load_yaml
    with open(file_path, "r", encoding="utf-8") as file:
         ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/finetune_config.yaml'


In [ ]:
!ls -lah /content/qwen3_1.7b_lora/

In [ ]:
!ls -lah /content/qwen3_1.7b_lora/epoch_1_step_1365/model/

In [ ]:
import json

path = "/content/drive/MyDrive/IFT/qwen3_1.7b_lora/epoch_1_step_1365/model"

with open(f"{path}/adapter_config.json") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

In [ ]:
!pip install -U torchao

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "Qwen/Qwen3-1.7B-Base"
adapter_path = "/content/qwen3_1.7b_lora/epoch_1_step_1365/model"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

prompt = """Write a research paper introduction based on this context.

Context: This study investigates efficient training methods for large language models.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
import json

with open("/content/test.jsonl", "r") as f:
    example = json.loads(f.readline())

print("Instruction:", example["instruction"])
print("Input:", example["input"])
print("Expected output:", example["output"])

In [ ]:
prompt = f"""Instruction: {example['instruction']}

Input: {example['input']}

Expected output:"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=700,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("GENERATED OUTPUT:\n", generated)